In [43]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


# 1. Setup paths
geodata_path = Path.cwd().parent / "data/geo/raw"
shapefiles = list(geodata_path.glob("**/*.shp"))

# 2. Settings
TARGET_EPSG = 32615  # UTM Zone 15N
final_gdfs = {}

print(f"{'Filename':<30} | {'Rows':<6} | {'CRS Status'}")
print("-" * 60)

# 3. Single-pass processing
for file_path in shapefiles:
    name = file_path.stem
    
    # Load and immediately re-project to avoid redundant memory usage
    gdf = gpd.read_file(file_path).to_crs(epsg=TARGET_EPSG)
    
    # Add Area Columns (Calculated once since both use same base math)
    sq_meters = gdf.geometry.area
    gdf['area_m2'] = sq_meters
    gdf['area_km2'] = sq_meters / 1_000_000
    
    # Store in dictionary
    final_gdfs[name] = gdf
    
    # Inline Progress Check
    print(f"{name:<30} | {len(gdf):<6} | Unified to {TARGET_EPSG}")

# 4. Verify specific layer
if 'mo_vest_20' in final_gdfs:
    print("\nVerification for mo_vest_20:")
    print(final_gdfs['mo_vest_20'][['area_m2', 'area_km2']].head())

Filename                       | Rows   | CRS Status
------------------------------------------------------------
mo_2010_cnty_bound             | 115    | Unified to 32615
mo_vest_16                     | 3324   | Unified to 32615
mo_cnty_2020_bound             | 115    | Unified to 32615
mo_vest_20                     | 3733   | Unified to 32615
mo_2024_gen_all_prec           | 3333   | Unified to 32615
mo_2024_gen_cong_prec          | 3357   | Unified to 32615
mo_2024_gen_sldl_prec          | 3573   | Unified to 32615
mo_2024_gen_sldu_prec          | 3335   | Unified to 32615

Verification for mo_vest_20:
        area_m2   area_km2
0  9.149077e+07  91.490775
1  3.318367e+07  33.183667
2  6.148763e+06   6.148763
3  9.835783e+06   9.835783
4  2.108673e+06   2.108673


In [42]:
#  https://github.com/PublicI/us-polling-places/tree/update-2020/data POLLING PLACE DATA 2020

In [ ]:

# 1. Load the data
df = pd.read_excel("../data/raw/10-6-20 Polling places.xls")

# 2. Get top 25 unique addresses
# Change .head(25) to .head(100) or remove it entirely to run the whole file
unique_locations = df.drop_duplicates(subset=['LOCATION_ADDRESS']).head(25).copy()

# 3. Initialize the geocoder
# OpenStreetMap (Nominatim) is free but requires a 1-second delay between requests
geolocator = Nominatim(user_agent="mo_voting_sample_analysis")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

print("Geocoding addresses... (This may take a moment)")
unique_locations['location'] = unique_locations['LOCATION_ADDRESS'].apply(geocode)

# 4. Extract latitude and longitude
unique_locations['lat'] = unique_locations['location'].apply(lambda loc: loc.latitude if loc else None)
unique_locations['lon'] = unique_locations['location'].apply(lambda loc: loc.longitude if loc else None)

# Filter out any addresses the geocoder couldn't find
clean_locations = unique_locations.dropna(subset=['lat', 'lon'])

# 5. Create the Spatial GeoDataFrame
gdf = gpd.GeoDataFrame(
    clean_locations, 
    geometry=gpd.points_from_xy(clean_locations.lon, clean_locations.lat),
    crs="EPSG:4326" # Standard WGS84 projection for web maps
)

# Optional: Drop the raw 'location' object column before saving
gdf = gdf.drop(columns=['location'])

# 6. Save your final spatial files!
gdf.to_csv("actual_geocoded_polling_places.csv", index=False)
gdf.to_file("actual_geocoded_polling_places.geojson", driver="GeoJSON")

print(f"Successfully geocoded {len(gdf)} addresses!")

Geocoding addresses... (This may take a moment)
Successfully geocoded 18 addresses!


In [51]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
from tqdm import tqdm
import time

# Enable progress bar
tqdm.pandas()

df = pd.read_excel("../data/raw/10-6-20 Polling places.xls")
unique_locations = df.drop_duplicates(subset=['LOCATION_ADDRESS']).copy()
print(f"Starting geocoding for {len(unique_locations)} unique addresses...")

# 1. ADDED TIMEOUT = 10 SECONDS
geolocator = Nominatim(user_agent="mo_voting_full_analysis", timeout=10)

# 2. CUSTOM SAFE GEOCODER FUNCTION
def safe_geocode(address, attempt=1, max_attempts=3):
    try:
        # Sleep for 1.1 seconds to respect OpenStreetMap's terms of service
        time.sleep(1.1) 
        return geolocator.geocode(address)
    except (GeocoderTimedOut, GeocoderUnavailable):
        if attempt <= max_attempts:
            time.sleep(2) # Wait a bit longer if the server is busy
            return safe_geocode(address, attempt=attempt+1, max_attempts=max_attempts)
        return None
    except Exception:
        return None

# 3. APPLY WITH PROGRESS BAR
unique_locations['location'] = unique_locations['LOCATION_ADDRESS'].progress_apply(safe_geocode)

# Extract coordinates
unique_locations['lat'] = unique_locations['location'].apply(lambda loc: loc.latitude if loc else None)
unique_locations['lon'] = unique_locations['location'].apply(lambda loc: loc.longitude if loc else None)

# Filter and create GeoDataFrame...
clean_locations = unique_locations.dropna(subset=['lat', 'lon'])
gdf = gpd.GeoDataFrame(clean_locations, geometry=gpd.points_from_xy(clean_locations.lon, clean_locations.lat), crs="EPSG:4326")
gdf = gdf.drop(columns=['location'])

gdf.to_csv("../data/raw/MO_2020_Polling_Geocoded_Full.csv", index=False)
gdf.to_file("../data/raw/MO_2020_Polling_Geocoded_Full.geojson", driver="GeoJSON")

print(f"\nDone! Successfully geocoded {len(gdf)} out of {len(unique_locations)} addresses.")

Starting geocoding for 2233 unique addresses...


100%|██████████| 2233/2233 [1:10:13<00:00,  1.89s/it]



Done! Successfully geocoded 1429 out of 2233 addresses.


In [ ]:
# Load the data
df = pd.read_excel("../data/raw/10-6-20 Polling places.xls")

# Calculate Total Polling Locations per County
unique_locations = df.drop_duplicates(subset=['COUNTY', 'LOCATION_ADDRESS'])
county_counts = unique_locations.groupby('COUNTY').size().reset_index(name='Polling_Place_Count')
county_counts = county_counts.sort_values(by='Polling_Place_Count', ascending=False)

# Calculate Average Precincts per Polling Location (Resource Strain)
precincts_per_location = df.groupby(['COUNTY', 'LOCATION_ADDRESS'])['PRECINCT'].nunique().reset_index(name='Precincts_Served')
avg_precincts_served = precincts_per_location.groupby('COUNTY')['Precincts_Served'].mean().reset_index(name='Avg_Precincts_per_Location')
avg_precincts_served = avg_precincts_served.sort_values(by='Avg_Precincts_per_Location', ascending=False)

print(avg_precincts_served.head(10))